In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [2]:
order_details = pd.read_csv('data/order_details.csv', encoding='windows-1251')
products = pd.read_csv('data/products.csv', encoding='windows-1251')
categories = pd.read_csv('data/categories.csv', encoding='windows-1251')

In [3]:
order_details

,orderID,productID,unitPrice,quantity,discount
0,10248,11,14.00,12,0.00
1,10248,42,9.80,10,0.00
2,10248,72,34.80,5,0.00
3,10249,14,18.60,9,0.00
4,10249,51,42.40,40,0.00
...,...,...,...,...,...
2150,11077,64,33.25,2,0.03
2151,11077,66,17.00,1,0.00
2152,11077,73,15.00,2,0.01
2153,11077,75,7.75,4,0.00


In [4]:
data = order_details.merge(products, how='inner', on='productID')
df = data.merge(categories, how='inner', on='categoryID')
df = df[['productName', 'categoryName', 'unitPrice_x', 'quantity']]
df

,productName,categoryName,unitPrice_x,quantity
0,Queso Cabrales,Dairy Products,14.00,12
1,Singaporean Hokkien Fried Mee,Grains & Cereals,9.80,10
2,Mozzarella di Giovanni,Dairy Products,34.80,5
3,Tofu,Produce,18.60,9
4,Manjimup Dried Apples,Produce,42.40,40
...,...,...,...,...
2150,Wimmers gute Semmelknцdel,Grains & Cereals,33.25,2
2151,Louisiana Hot Spiced Okra,Condiments,17.00,1
2152,Rцd Kaviar,Seafood,15.00,2
2153,Rhцnbrдu Klosterbier,Beverages,7.75,4


In [5]:
goods = [
    'Chai',
    'Cфte de Blaye',
    "Sir Rodney's Marmalade", 
    'Gorgonzola Telino', 
    'Ikura', 
    'Louisiana Hot Spiced Okra'
    ]

df_selected = df[df['productName'].isin(goods)].reset_index(drop=True)

df_selected

,productName,categoryName,unitPrice_x,quantity
0,Sir Rodney's Marmalade,Confections,64.8,40
1,Gorgonzola Telino,Dairy Products,10.0,20
2,Sir Rodney's Marmalade,Confections,64.8,6
3,Gorgonzola Telino,Dairy Products,10.0,40
4,Ikura,Seafood,24.8,24
...,...,...,...,...
165,Chai,Beverages,18.0,40
166,Gorgonzola Telino,Dairy Products,12.5,20
167,Ikura,Seafood,31.0,1
168,Sir Rodney's Marmalade,Confections,81.0,1


In [6]:
stats = df_selected.groupby('productName').agg({
    'unitPrice_x': ['count', 'mean', 'std', 'min', 'max'],
    'quantity': ['sum', 'mean']
})

stats

unitPrice_x                                       \
                                count        mean        std    min    max   
productName                                                                  
Chai                               38   17.147368   1.551065   14.4   18.0   
Cфte de Blaye                      24  245.933333  25.377338  210.8  263.5   
Gorgonzola Telino                  51   11.666667   1.190238   10.0   12.5   
Ikura                              33   29.684848   2.573922   24.8   31.0   
Louisiana Hot Spiced Okra           8   15.300000   1.817376   13.6   17.0   
Sir Rodney's Marmalade             16   75.937500   7.755160   64.8   81.0   

                          quantity             
                               sum       mean  
productName                                    
Chai                           828  21.789474  
Cфte de Blaye                  623  25.958333  
Gorgonzola Telino             1397  27.392157  
Ikura                          742  22.484848  
Louisiana Hot Spiced Okra      239  29.875000  
Sir Rodney's Marmalade         313  19.562500

In [7]:
demand_params = {}

for product in goods:
    subdata = df_selected[df_selected['productName'] == product].copy()
    subdata = subdata[subdata['quantity'] > 0]

    X = subdata[['unitPrice_x']]
    y = subdata['quantity']

    lr = LinearRegression()
    lr.fit(X, y)

    a = lr.intercept_
    b = lr.coef_[0]
    r2 = r2_score(y, lr.predict(X))

    mean_price = subdata['unitPrice_x'].mean()
    cost = mean_price * 0.6

    demand_params[product] = {
        'a': a,
        'b': b,
        'cost': cost,
        'mean_price': mean_price,
        'max_price': subdata['unitPrice_x'].max()
    }

In [8]:
for product in demand_params:
    subset = df_selected[df_selected['productName'] == product]
    demand_params[product]['q_mean'] = float(subset['quantity'].mean())

for product, data in demand_params.items():
    p_mean = data['mean_price']
    q_mean = data['q_mean']
    
    if product == 'Côte de Blaye':
        elasticity = -1.2
    elif product in ['Chai', 'Louisiana Hot Spiced Okra']:
        elasticity = -2.5
    else:
        elasticity = -2.0

    b_calibrated = elasticity * (q_mean / p_mean)
    a_calibrated = q_mean - (b_calibrated * p_mean)

    data['a'] = float(a_calibrated)
    data['b'] = float(b_calibrated)
    data['elasticity'] = elasticity
    

In [9]:
def calculate_total_profit(prices, models):
    total_profit = 0
    product_names = list(models.keys())

    for i, product in enumerate(product_names):
        P = prices[i]
        a = models[product]['a']
        b = models[product]['b']
        C = models[product]['cost']

        demand = a + b * P
        demand = max(0, demand)

        profit_i = (P - C) * demand
        total_profit += profit_i
        print( (P, demand, profit_i, total_profit) )

    return total_profit

In [10]:
historical_prices = [demand_params[p]['mean_price'] for p in demand_params.keys()]
historical_profit = calculate_total_profit(historical_prices, demand_params)
print(historical_profit)

(np.float64(17.147368421052633), np.float64(21.789473684210535), np.float64(149.45285318559567), np.float64(149.45285318559567))
(np.float64(245.9333333333333), np.float64(25.958333333333336), np.float64(2553.607777777778), np.float64(2703.060630963374))
(np.float64(75.9375), np.float64(19.5625), np.float64(594.2109375), np.float64(3297.271568463374))
(np.float64(11.666666666666666), np.float64(27.392156862745097), np.float64(127.83006535947713), np.float64(3425.101633822851))
(np.float64(29.684848484848484), np.float64(22.484848484848484), np.float64(266.9837281910009), np.float64(3692.085362013852))
(np.float64(15.3), np.float64(29.875), np.float64(182.83500000000004), np.float64(3874.920362013852))
3874.920362013852
